# 동물상 분류 모델 데이터
- 최소 데이터: 20개 클래스 20장씩


## 1. 데이터 수집

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import numpy as np
import time

In [ ]:
from bing_image_downloader import downloader

downloader.download(
    query="알파카상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="공룡상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="원숭이상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="오리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="코알라상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="늑대상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="햄스터상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="꽃돼지상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="너구리상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

downloader.download(
    query="말상",             # 검색어
    limit=50,
    output_dir="./dataset/raw",     # 파일 저장 위치
    adult_filter_off=True,
    force_replace=False,
    timeout=30 
)

## 2. 얼굴 추출
- 1) dataset/raw(클래스당 70장)에서 얼굴 크롭을 돌린 다음
- 2) “괜찮은 결과” 위주로 클래스당 50장만 골라서 dataset/selected50/<클래스>/에 저장하는 코드


In [ ]:
import cv2, numpy as np, random, shutil
from pathlib import Path
from collections import defaultdict
from insightface.app import FaceAnalysis
from insightface.utils.face_align import norm_crop
from sklearn.cluster import DBSCAN

# ---------- 유니코드 경로 읽기/쓰기 ----------
def imread_unicode(path: Path):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

def imwrite_unicode_jpg(path: Path, img):
    path.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(".jpg", img)
    if not ok:
        raise RuntimeError(f"imencode failed: {path}")
    buf.tofile(str(path))

def center_square_resize(img, size=112):
    h, w = img.shape[:2]
    s = min(h, w)
    y1 = (h - s) // 2
    x1 = (w - s) // 2
    crop = img[y1:y1+s, x1:x1+s]
    return cv2.resize(crop, (size, size), interpolation=cv2.INTER_AREA)

def l2_normalize(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x)
    return x / (n + 1e-12)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# ======================
# 설정
# ======================
RAW_DIR = Path("dataset/raw")             # 클래스당 70장
SELECTED_DIR = Path("dataset/selected50") # 클래스당 50장 저장
TRAIN_DIR = Path("dataset/train")
TEST_DIR  = Path("dataset/test")

TARGET_PER_CLASS = 50
TRAIN_RATIO = 0.8                         # 50장이면 대략 40/10
SEED = 42
COPY_MODE = True                          # True=복사, False=이동 (raw는 보통 복사 추천)

# DBSCAN(사람 추정 그룹) 파라미터
DBSCAN_EPS = 0.35
DBSCAN_MIN_SAMPLES = 2

random.seed(SEED)
np.random.seed(SEED)

# ✅ 검출률 높이기(너 환경 기준)
app = FaceAnalysis(name="buffalo_l")
app.prepare(ctx_id=0, det_size=(1280,1280), det_thresh=0.1)

def copy_or_move(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if COPY_MODE:
        shutil.copy2(src, dst)
    else:
        shutil.move(str(src), str(dst))

def detect_best_face(app, img):
    # 1) 원본
    faces = app.get(img)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img

    # 2) padding
    h, w = img.shape[:2]
    pad = int(0.25 * max(h, w))
    img_pad = cv2.copyMakeBorder(img, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(0,0,0))
    faces = app.get(img_pad)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_pad

    # 3) upsample
    scale = 1.5
    img_up = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
    faces = app.get(img_up)
    if faces:
        face = max(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))
        return face, img_up

    return None, img

# ======================
# 1) raw(70) -> selected50(50) (얼굴 크롭/정렬 후 상위 50개 선택)
# ======================
SELECTED_DIR.mkdir(parents=True, exist_ok=True)

for cls_dir in [d for d in RAW_DIR.iterdir() if d.is_dir()]:
    imgs = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]
    if not imgs:
        print(f"[WARN] no images: {cls_dir.name}")
        continue

    random.shuffle(imgs)

    candidates = []  # (score, cropped112, src_path)
    for p in imgs:
        img = imread_unicode(p)
        if img is None:
            continue

        face, used_img = detect_best_face(app, img)

        if face is not None and getattr(face, "kps", None) is not None:
            out = norm_crop(used_img, face.kps)  # 112x112
            det_score = float(getattr(face, "det_score", 0.0))
            score = det_score + 1.0  # aligned 보너스
        else:
            out = center_square_resize(img, 112)
            score = 0.0  # fallback는 낮게

        candidates.append((score, out, p))

    if len(candidates) < TARGET_PER_CLASS:
        print(f"[WARN] {cls_dir.name}: candidates={len(candidates)} < target={TARGET_PER_CLASS} (그래도 진행)")
    candidates.sort(key=lambda x: x[0], reverse=True)
    selected = candidates[:TARGET_PER_CLASS]

    out_cls = SELECTED_DIR / cls_dir.name
    out_cls.mkdir(parents=True, exist_ok=True)

    for i, (score, out_img, src) in enumerate(selected, start=1):
        out_name = f"{cls_dir.name}_{i:03d}.jpg"
        imwrite_unicode_jpg(out_cls / out_name, out_img)

    print(f"✅ selected50 {cls_dir.name}: selected={len(selected)} (top_score={selected[0][0] if selected else None})")

print("\n✅ Step1 done:", SELECTED_DIR.resolve())